# ניתוח נתונים - Smart Task Manager

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

conn = sqlite3.connect('../backend/database.db')
tasks_df = pd.read_sql_query('SELECT * FROM tasks', conn)
users_df = pd.read_sql_query('SELECT user_id, full_name, role FROM users', conn)
conn.close()

print('סה"כ משימות:', len(tasks_df))
tasks_df.head()

## כמה משימות מכל סטטוס

In [ ]:
status_counts = tasks_df['status'].value_counts()
print(status_counts)

status_labels = {'open': 'פתוח', 'in_progress': 'בתהליך', 'closed': 'סגור'}
status_counts.index = [status_labels.get(s, s) for s in status_counts.index]

plt.figure(figsize=(6, 4))
status_counts.plot(kind='bar', color=['#4CAF50', '#FF9800', '#F44336'])
plt.title('התפלגות משימות לפי סטטוס')
plt.xlabel('סטטוס')
plt.ylabel('כמות')
plt.tight_layout()
plt.show()

## זמן סגירה ממוצע של משימה (בימים)

In [ ]:
closed_tasks = tasks_df[tasks_df['status'] == 'closed'].copy()

if 'due_date' in closed_tasks.columns and closed_tasks['due_date'].notna().any():
    closed_tasks['due_date'] = pd.to_datetime(closed_tasks['due_date'], errors='coerce')
    today = pd.Timestamp.today()
    closed_tasks['days_to_close'] = (today - closed_tasks['due_date']).dt.days
    avg_days = np.mean(closed_tasks['days_to_close'].dropna().values)
    print(f'זמן סגירה ממוצע: {avg_days:.1f} ימים')
else:
    print('אין נתוני תאריך יעד למשימות סגורות')

## התפלגות משימות לפי אחראי

In [ ]:
merged = tasks_df.merge(users_df, left_on='assigned_to', right_on='user_id', how='left')
merged['full_name'] = merged['full_name'].fillna('לא מוקצה')

tasks_per_user = merged.groupby('full_name').size().sort_values(ascending=False)
print(tasks_per_user)

plt.figure(figsize=(8, 5))
tasks_per_user.plot(kind='bar', color='#2196F3')
plt.title('התפלגות משימות לפי אחראי')
plt.xlabel('משתמש')
plt.ylabel('כמות משימות')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## התפלגות משימות לפי עדיפות

In [ ]:
priority_map = {1: 'נמוך', 2: 'בינוני', 3: 'גבוה'}
tasks_df['priority_label'] = tasks_df['priority'].map(priority_map)
priority_counts = tasks_df['priority_label'].value_counts()

plt.figure(figsize=(5, 5))
plt.pie(priority_counts, labels=priority_counts.index, autopct='%1.1f%%',
        colors=['#4CAF50', '#FF9800', '#F44336'])
plt.title('התפלגות משימות לפי עדיפות')
plt.show()